# OOF Predictions – Error Analysis

Analyze out-of-fold (OOF) predictions with **Phoneme Error Rate (PER)** as the primary metric.
PER = edit_distance(reference, hypothesis) / len(reference), computed at the IPA-character level.

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

In [ ]:
import pandas as pd
from pathlib import Path

# Set paths relative to the repository root
oof_rel_path = "outputs/submit-day/18-00-56_annoying-Liam-252/oof_predictions_best.parquet" # 'outputs/2026-03-20/04-37-34_climate_killing-Willem-209/oof_predictions_best.parquet'
meta_rel_path = 'data/audio_files_with_labels_clean.jsonl'

# Resolve repo root by walking up to a marker file
repo_root = Path.cwd().resolve()
for parent in [repo_root] + list(repo_root.parents):
    if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
        repo_root = parent
        break

# Construct exact absolute paths
OOF_PATH = repo_root / oof_rel_path
META_PATH = repo_root / meta_rel_path

# Strict check for OOF path
if not OOF_PATH.exists():
    raise FileNotFoundError(f"Strict loading failed. OOF file not found exactly at: {OOF_PATH}")
if not OOF_PATH.is_file():
    raise FileNotFoundError(f"OOF path exists but is not a file: {OOF_PATH}")

# Strict check for metadata path
if not META_PATH.exists():
    raise FileNotFoundError(f"Strict loading failed. Metadata file not found exactly at: {META_PATH}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Metadata path exists but is not a file: {META_PATH}")

print(f"Repo root: {repo_root}")
print(f"Using exact OOF file: {OOF_PATH}")
print(f"Using exact metadata file: {META_PATH}")

# Load data
preds = pd.read_parquet(OOF_PATH)
meta = pd.read_json(META_PATH, lines=True)

print('preds:', preds.shape)
print('meta:', meta.shape)
print(preds.head())

In [ ]:
# Join predictions to metadata
meta_cols = [
    'utterance_id', 'child_id', 'session_id', 'audio_duration_sec',
    'age_bucket', 'phonetic_text', 'source'
]
meta_use = meta[meta_cols].copy()

# Avoid column clashes with metadata (e.g., age_bucket)
preds = preds.drop(columns=['age_bucket'], errors='ignore')

# Basic join diagnostic
if meta_use['utterance_id'].duplicated().any():
    dup = meta_use[meta_use['utterance_id'].duplicated(keep=False)]
    print(f"Warning: duplicated utterance_id in metadata: {dup['utterance_id'].nunique()} ids")

# Merge on utterance_id + child_id for safety
# If you want a looser join, use on='utterance_id' and inspect mismatches.
df = preds.merge(meta_use, on=['utterance_id', 'child_id'], how='left')

missing_meta = df['audio_duration_sec'].isna().mean()
print(f"Missing metadata for {missing_meta:.2%} of rows")

df.head()

In [ ]:
# ── Compute PER (Phoneme Error Rate) ───────────────────────────────────
def edit_distance(a: list, b: list) -> int:
    """Levenshtein distance on two lists."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[0]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev if ca == cb else 1 + min(prev, dp[j - 1], cur)
            prev = cur
    return dp[-1]

# Normalise text
gt_str   = df['ground_truth'].astype('string').fillna('').str.strip()
pred_str = df['prediction'].astype('string').fillna('').str.strip()

# Character-level phoneme lists (strip spaces – spaces separate words, not phonemes)
gt_phones   = gt_str.apply(lambda s: list(s.replace(' ', '')))
pred_phones = pred_str.apply(lambda s: list(s.replace(' ', '')))

# Compute edit distance & PER
df['edit_dist'] = [edit_distance(g, p) for g, p in zip(gt_phones, pred_phones)]
df['gt_len']    = gt_phones.apply(len)
df['pred_len']  = pred_phones.apply(len)
df['per']       = np.where(df['gt_len'] > 0, df['edit_dist'] / df['gt_len'], np.nan)
df['is_correct'] = (gt_str == pred_str)
df['len_diff']  = df['pred_len'] - df['gt_len']  # positive = insertion-heavy

# Duration buckets
if df['audio_duration_sec'].notna().any():
    df['duration_bucket'] = pd.qcut(df['audio_duration_sec'], q=5, duplicates='drop')
else:
    df['duration_bucket'] = pd.NA

df['age_bucket'] = df['age_bucket'].fillna('unknown')

print(f"Overall PER          : {df['per'].mean():.4f}")
print(f"Median  PER          : {df['per'].median():.4f}")
print(f"Exact-match accuracy : {df['is_correct'].mean():.4f}")
print(f"Total utterances     : {len(df)}")

## 1 Overall Summary

In [ ]:
summary = pd.DataFrame({
    'metric': ['Utterances', 'Mean PER', 'Median PER', 'Exact-Match Accuracy',
               'Mean Edit Distance', 'Mean GT Length (phones)'],
    'value': [len(df), df['per'].mean(), df['per'].median(),
              df['is_correct'].mean(), df['edit_dist'].mean(), df['gt_len'].mean()],
})
summary.style.format({'value': '{:.4f}'}).hide(axis='index')

In [ ]:
# PER distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['per'].dropna(), bins=50, edgecolor='k', alpha=0.7)
axes[0].axvline(df['per'].mean(), color='red', ls='--', label=f"mean={df['per'].mean():.3f}")
axes[0].axvline(df['per'].median(), color='orange', ls='--', label=f"median={df['per'].median():.3f}")
axes[0].set_title('PER Distribution')
axes[0].set_xlabel('Phoneme Error Rate')
axes[0].legend()

# Cumulative distribution
sorted_per = np.sort(df['per'].dropna())
axes[1].plot(sorted_per, np.linspace(0, 1, len(sorted_per)))
axes[1].set_title('PER Cumulative Distribution')
axes[1].set_xlabel('PER')
axes[1].set_ylabel('Fraction of utterances')
axes[1].axhline(0.5, color='grey', ls=':', alpha=0.5)
axes[1].axhline(0.9, color='grey', ls=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 2 PER by Age Group

In [ ]:
def age_sort_key(x):
    if x == 'unknown':
        return (999,)
    try:
        return (float(str(x).split('-')[0]),)
    except Exception:
        return (998,)

age_order = sorted(df['age_bucket'].unique(), key=age_sort_key)

age_stats = (
    df.groupby('age_bucket')
      .agg(
          n=('per', 'size'),
          mean_per=('per', 'mean'),
          median_per=('per', 'median'),
          std_per=('per', 'std'),
          exact_match=('is_correct', 'mean'),
      )
      .reindex(age_order)
)
age_stats.style.format({
    'mean_per': '{:.4f}', 'median_per': '{:.4f}',
    'std_per': '{:.4f}', 'exact_match': '{:.4f}',
}).background_gradient(subset=['mean_per'], cmap='Reds')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sns.barplot(data=age_stats.reset_index(), x='age_bucket', y='mean_per',
            order=age_order, ax=axes[0], palette='rocket')
axes[0].set_title('Mean PER by Age Group')
axes[0].set_xlabel('Age Bucket')
axes[0].set_ylabel('Mean PER')
axes[0].tick_params(axis='x', rotation=45)

# Box plot
sns.boxplot(data=df, x='age_bucket', y='per', order=age_order, ax=axes[1],
            showfliers=False, palette='rocket')
axes[1].set_title('PER Distribution by Age Group')
axes[1].set_xlabel('Age Bucket')
axes[1].set_ylabel('PER')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3 PER by Audio Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Duration distribution
sns.histplot(df['audio_duration_sec'].dropna(), bins=60, ax=axes[0], kde=True)
axes[0].set_title('Audio Duration Distribution')
axes[0].set_xlabel('Duration (sec)')

# Scatter: PER vs duration
sample = df.dropna(subset=['audio_duration_sec', 'per']).sample(min(5000, len(df)), random_state=42)
axes[1].scatter(sample['audio_duration_sec'], sample['per'], alpha=0.15, s=8)
axes[1].set_title('PER vs Audio Duration (sampled)')
axes[1].set_xlabel('Duration (sec)')
axes[1].set_ylabel('PER')

plt.tight_layout()
plt.show()

In [ ]:
if df['duration_bucket'].notna().any():
    dur_stats = (
        df.groupby('duration_bucket', observed=True)
          .agg(
              n=('per', 'size'),
              mean_per=('per', 'mean'),
              median_per=('per', 'median'),
              exact_match=('is_correct', 'mean'),
          )
          .reset_index()
    )
    display(dur_stats.style.format({
        'mean_per': '{:.4f}', 'median_per': '{:.4f}', 'exact_match': '{:.4f}',
    }).background_gradient(subset=['mean_per'], cmap='Reds'))

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=dur_stats, x='duration_bucket', y='mean_per', ax=ax, palette='mako')
    ax.set_title('Mean PER by Duration Bucket (Quantiles)')
    ax.set_xlabel('Duration Bucket')
    ax.set_ylabel('Mean PER')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
## 4 PER by Utterance Length (number of ground-truth phonemes)

# Bin GT phoneme length
df['gt_len_bin'] = pd.cut(df['gt_len'], bins=[0, 2, 4, 6, 8, 10, 15, 20, 100],
                          labels=['1-2','3-4','5-6','7-8','9-10','11-15','16-20','21+'])

len_stats = (
    df.groupby('gt_len_bin', observed=True)
      .agg(n=('per','size'), mean_per=('per','mean'), median_per=('per','median'),
           exact_match=('is_correct','mean'))
      .reset_index()
)
display(len_stats.style.format({
    'mean_per':'{:.4f}','median_per':'{:.4f}','exact_match':'{:.4f}',
}).background_gradient(subset=['mean_per'], cmap='Reds'))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=len_stats, x='gt_len_bin', y='mean_per', ax=ax, palette='flare')
ax.set_title('Mean PER by Ground-Truth Phoneme Count')
ax.set_xlabel('# Phonemes in GT')
ax.set_ylabel('Mean PER')
plt.tight_layout()
plt.show()

In [ ]:
## 5 Age Group x Duration Cross-Section Heatmap

if df['duration_bucket'].notna().any():
    heat = (
        df.pivot_table(values='per', index='age_bucket',
                       columns='duration_bucket', aggfunc='mean', observed=True)
          .reindex(age_order)
    )
    plt.figure(figsize=(10, 4))
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=.5)
    plt.title('Mean PER: Age Group x Duration Bucket')
    plt.xlabel('Duration Bucket')
    plt.ylabel('Age Bucket')
    plt.tight_layout()
    plt.show()

In [ ]:
fold_stats = (
    df.groupby('fold')
      .agg(n=('per','size'), mean_per=('per','mean'),
           median_per=('per','median'), exact_match=('is_correct','mean'))
      .reset_index()
)
display(fold_stats)

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(data=fold_stats, x='fold', y='mean_per', ax=ax, palette='crest')
ax.set_title('Mean PER by Fold')
ax.set_ylabel('Mean PER')
plt.tight_layout()
plt.show()

## 7 PER by Data Source

In [ ]:
if 'source' in df.columns and df['source'].notna().any():
    src_stats = (
        df.groupby('source')
          .agg(n=('per','size'), mean_per=('per','mean'),
               median_per=('per','median'), exact_match=('is_correct','mean'))
          .sort_values('mean_per', ascending=False)
          .reset_index()
    )
    display(src_stats.style.format({
        'mean_per':'{:.4f}','median_per':'{:.4f}','exact_match':'{:.4f}',
    }).background_gradient(subset=['mean_per'], cmap='Reds'))
else:
    print("No 'source' column available")

## 8 Child-Level Analysis

## 9 Most Confused Phoneme Pairs (Character-Level)

Align GT and predicted phonemes via edit-distance back-trace and count the most frequent substitutions, insertions, and deletions.

In [ ]:
child_stats = (
    df.groupby('child_id')
      .agg(n=('per','size'), mean_per=('per','mean'), exact_match=('is_correct','mean'))
      .reset_index()
)
child_stats_20 = child_stats[child_stats['n'] >= 20].copy()
print(f"Children total: {len(child_stats)}")
print(f"Children with >=20 samples: {len(child_stats_20)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(child_stats_20['mean_per'], bins=30, edgecolor='k', alpha=0.7)
axes[0].set_title('Child-Level Mean PER Distribution (n>=20)')
axes[0].set_xlabel('Mean PER')

# Worst children
worst = child_stats_20.nlargest(15, 'mean_per')
sns.barplot(data=worst, y='child_id', x='mean_per', ax=axes[1], palette='Reds_r')
axes[1].set_title('Top 15 Worst Children by PER')
axes[1].set_xlabel('Mean PER')
axes[1].set_ylabel('')
axes[1].set_yticklabels([t.get_text()[:12] + '...' for t in axes[1].get_yticklabels()])

plt.tight_layout()
plt.show()

In [ ]:
def alignment_ops(a: list, b: list):
    """Return list of (op, gt_char, pred_char) from Levenshtein alignment."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    # Backtrace
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            i -= 1; j -= 1  # match
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(('sub', a[i-1], b[j-1]))
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(('del', a[i-1], '_'))
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            ops.append(('ins', '_', b[j-1]))
            j -= 1
        else:
            break  # safety
    return ops

# Sample a subset for speed (alignment is O(n^2) per utterance)
errors_df = df[~df['is_correct']].copy()
sample_errors = errors_df.sample(min(10000, len(errors_df)), random_state=42)

sub_counts = Counter()
del_counts = Counter()
ins_counts = Counter()

for _, row in sample_errors.iterrows():
    g = list(str(row['ground_truth']).replace(' ', ''))
    p = list(str(row['prediction']).replace(' ', ''))
    for op, gc, pc in alignment_ops(g, p):
        if op == 'sub':
            sub_counts[(gc, pc)] += 1
        elif op == 'del':
            del_counts[gc] += 1
        elif op == 'ins':
            ins_counts[pc] += 1

print(f"Analyzed {len(sample_errors)} error utterances")

In [ ]:
# Top substitutions
sub_df = pd.DataFrame(sub_counts.most_common(25), columns=['pair', 'count'])
sub_df[['gt_phone', 'pred_phone']] = pd.DataFrame(sub_df['pair'].tolist(), index=sub_df.index)
sub_df = sub_df.drop(columns='pair')

print("-- Top 25 Substitutions (GT -> Pred) --")
display(sub_df)

In [ ]:
# Top deletions & insertions side by side
del_df = pd.DataFrame(del_counts.most_common(15), columns=['phone', 'deletions'])
ins_df = pd.DataFrame(ins_counts.most_common(15), columns=['phone', 'insertions'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=del_df, x='deletions', y='phone', ax=axes[0], palette='Oranges_r')
axes[0].set_title('Top 15 Deleted Phonemes (in GT but not predicted)')

sns.barplot(data=ins_df, x='insertions', y='phone', ax=axes[1], palette='Blues_r')
axes[1].set_title('Top 15 Inserted Phonemes (predicted but not in GT)')

plt.tight_layout()
plt.show()

## 10 Phoneme Confusion Matrix (Top Phonemes)

In [ ]:
# Build a small confusion matrix for the most common GT phonemes in substitutions
gt_freq = Counter()
for (gc, _), c in sub_counts.items():
    gt_freq[gc] += c
top_phones = [p for p, _ in gt_freq.most_common(15)]

cm = pd.DataFrame(0, index=top_phones, columns=top_phones)
for (gc, pc), c in sub_counts.items():
    if gc in top_phones and pc in top_phones:
        cm.loc[gc, pc] += c

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', linewidths=.3)
plt.title('Phoneme Confusion Matrix (substitutions only, top 15)')
plt.xlabel('Predicted Phoneme')
plt.ylabel('Ground-Truth Phoneme')
plt.tight_layout()
plt.show()

## 11 Error-Type Breakdown

How much of the total PER comes from **substitutions**, **deletions**, and **insertions**?
Computed on all utterances (not just the sampled error set) using per-utterance alignment.

In [ ]:
# ── Per-utterance sub / del / ins counts (full dataset) ───────────────
# alignment_ops is already defined above; this adds columns to df so we
# can slice the breakdown by age, duration, fold, etc.
n_sub_list, n_del_list, n_ins_list = [], [], []

for _, row in df.iterrows():
    g = list(str(row['ground_truth']).replace(' ', ''))
    p = list(str(row['prediction']).replace(' ', ''))
    n_s = n_d = n_i = 0
    for op, _, _ in alignment_ops(g, p):
        if op == 'sub': n_s += 1
        elif op == 'del': n_d += 1
        elif op == 'ins': n_i += 1
    n_sub_list.append(n_s)
    n_del_list.append(n_d)
    n_ins_list.append(n_i)

df['n_sub'] = n_sub_list
df['n_del'] = n_del_list
df['n_ins'] = n_ins_list

# Contribution to PER as a fraction of GT length
df['sub_rate'] = np.where(df['gt_len'] > 0, df['n_sub'] / df['gt_len'], np.nan)
df['del_rate'] = np.where(df['gt_len'] > 0, df['n_del'] / df['gt_len'], np.nan)
df['ins_rate'] = np.where(df['gt_len'] > 0, df['n_ins'] / df['gt_len'], np.nan)

total_s = df['n_sub'].sum()
total_d = df['n_del'].sum()
total_i = df['n_ins'].sum()
total_e = total_s + total_d + total_i

print(f"Total errors  : {total_e:,}")
print(f"Substitutions : {total_s:,}  ({total_s/total_e:.1%})")
print(f"Deletions     : {total_d:,}  ({total_d/total_e:.1%})")
print(f"Insertions    : {total_i:,}  ({total_i/total_e:.1%})")

In [ ]:
# ── Overall pie + mean rate per utterance ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie: share of total edit operations
axes[0].pie(
    [total_s, total_d, total_i],
    labels=['Substitutions', 'Deletions', 'Insertions'],
    autopct='%1.1f%%',
    colors=['#e05c5c', '#f5a623', '#4a90d9'],
    startangle=140,
)
axes[0].set_title('Share of Total Edit Operations')

# Mean rate per utterance (as fraction of GT length)
mean_rates = pd.DataFrame({
    'Error Type': ['Substitutions', 'Deletions', 'Insertions'],
    'Mean Rate': [df['sub_rate'].mean(), df['del_rate'].mean(), df['ins_rate'].mean()],
})
axes[1].bar(mean_rates['Error Type'], mean_rates['Mean Rate'],
            color=['#e05c5c', '#f5a623', '#4a90d9'], edgecolor='k', alpha=0.85)
axes[1].set_title('Mean Error Rate per Utterance\n(normalised by GT length)')
axes[1].set_ylabel('Rate (fraction of GT phonemes)')
for bar, val in zip(axes[1].patches, mean_rates['Mean Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Error-type breakdown by age group (stacked bar) ───────────────────
age_err = (
    df.groupby('age_bucket')[['sub_rate', 'del_rate', 'ins_rate']]
      .mean()
      .reindex(age_order)
      .rename(columns={'sub_rate': 'Substitution', 'del_rate': 'Deletion', 'ins_rate': 'Insertion'})
)

display(age_err.style.format('{:.4f}').background_gradient(cmap='Reds'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar
age_err.plot(
    kind='bar', stacked=True, ax=axes[0],
    color=['#e05c5c', '#f5a623', '#4a90d9'], edgecolor='k', alpha=0.88,
)
axes[0].set_title('Mean Error Rate by Type — Age Group')
axes[0].set_xlabel('Age Bucket')
axes[0].set_ylabel('Mean Rate (fraction of GT phones)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(loc='upper right')

# 100 % stacked (proportions)
age_err_norm = age_err.div(age_err.sum(axis=1), axis=0) * 100
age_err_norm.plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#e05c5c', '#f5a623', '#4a90d9'], edgecolor='k', alpha=0.88,
)
axes[1].set_title('Error Type Composition (%) — Age Group')
axes[1].set_xlabel('Age Bucket')
axes[1].set_ylabel('% of total errors')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Error-type breakdown by duration bucket ───────────────────────────
if df['duration_bucket'].notna().any():
    dur_err = (
        df.groupby('duration_bucket', observed=True)[['sub_rate', 'del_rate', 'ins_rate']]
          .mean()
          .rename(columns={'sub_rate': 'Substitution', 'del_rate': 'Deletion', 'ins_rate': 'Insertion'})
    )
    display(dur_err.style.format('{:.4f}').background_gradient(cmap='Reds'))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    dur_err.plot(
        kind='bar', stacked=True, ax=axes[0],
        color=['#e05c5c', '#f5a623', '#4a90d9'], edgecolor='k', alpha=0.88,
    )
    axes[0].set_title('Mean Error Rate by Type — Duration Bucket')
    axes[0].set_xlabel('Duration Bucket')
    axes[0].set_ylabel('Mean Rate (fraction of GT phones)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(loc='upper left')

    dur_err_norm = dur_err.div(dur_err.sum(axis=1), axis=0) * 100
    dur_err_norm.plot(
        kind='bar', stacked=True, ax=axes[1],
        color=['#e05c5c', '#f5a623', '#4a90d9'], edgecolor='k', alpha=0.88,
    )
    axes[1].set_title('Error Type Composition (%) — Duration Bucket')
    axes[1].set_xlabel('Duration Bucket')
    axes[1].set_ylabel('% of total errors')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(loc='upper left')

    plt.tight_layout()
    plt.show()

## 11 Insertion / Deletion Bias

Does the model systematically produce shorter or longer transcriptions?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Length difference histogram
axes[0].hist(df['len_diff'].dropna(), bins=range(-15, 16), edgecolor='k', alpha=0.7)
axes[0].axvline(0, color='red', ls='--')
axes[0].set_title('Prediction Length - GT Length')
axes[0].set_xlabel('Delta phonemes (positive = insertions)')
axes[0].set_ylabel('Count')

# Mean length diff by age
len_age = df.groupby('age_bucket')['len_diff'].mean().reindex(age_order)
len_age.plot.bar(ax=axes[1], color='steelblue')
axes[1].axhline(0, color='red', ls='--')
axes[1].set_title('Mean Length Diff by Age Group')
axes[1].set_ylabel('Mean Delta phonemes')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Overall mean length diff: {df['len_diff'].mean():.3f}")
print(f"Fraction of predictions shorter than GT: {(df['len_diff'] < 0).mean():.3f}")
print(f"Fraction of predictions longer  than GT: {(df['len_diff'] > 0).mean():.3f}")

## 12 Worst Utterances

Inspect the utterances with the highest PER to find systematic failure patterns.

In [ ]:
# Show worst 25 utterances (PER > 0, sorted desc)
worst_utts = (
    df[df['per'] > 0]
      .nlargest(25, 'per')
      [['utterance_id','child_id','age_bucket','audio_duration_sec',
        'gt_len','pred_len','edit_dist','per','ground_truth','prediction']]
)
display(worst_utts.style.format({'per':'{:.3f}','audio_duration_sec':'{:.2f}'})
        .background_gradient(subset=['per'], cmap='Reds'))

## 13 Complete Failures (PER >= 1.0)

Utterances where the model got **nothing** right.

In [ ]:
total_fail = df[df['per'] >= 1.0]
print(f"Utterances with PER >= 1.0: {len(total_fail)} ({len(total_fail)/len(df):.2%})")

if len(total_fail) > 0:
    fail_by_age = total_fail.groupby('age_bucket').size().reindex(age_order, fill_value=0)
    total_by_age = df.groupby('age_bucket').size().reindex(age_order, fill_value=0)
    fail_rate = (fail_by_age / total_by_age).fillna(0)

    fig, ax = plt.subplots(figsize=(8, 4))
    fail_rate.plot.bar(ax=ax, color='crimson', alpha=0.8)
    ax.set_title('Fraction of Complete Failures (PER >= 1.0) by Age Group')
    ax.set_ylabel('Failure Rate')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # What do these failures look like?
    display(total_fail.sample(min(10, len(total_fail)), random_state=42)
            [['age_bucket','audio_duration_sec','gt_len','ground_truth','prediction']])

## 14 Diagnostic Summary

In [ ]:
print("=" * 60)
print("DIAGNOSTIC SUMMARY")
print("=" * 60)
print(f"Overall Mean PER       : {df['per'].mean():.4f}")
print(f"Overall Median PER     : {df['per'].median():.4f}")
print(f"Exact-Match Accuracy   : {df['is_correct'].mean():.4f}")
print(f"Total Utterances       : {len(df)}")
print()

# Error type breakdown
if 'n_sub' in df.columns:
    _s = df['n_sub'].sum()
    _d = df['n_del'].sum()
    _i = df['n_ins'].sum()
    _t = _s + _d + _i
    print(f"Error breakdown (total ops = {_t:,}):")
    print(f"  Substitutions : {_s:,}  ({_s/_t:.1%})")
    print(f"  Deletions     : {_d:,}  ({_d/_t:.1%})")
    print(f"  Insertions    : {_i:,}  ({_i/_t:.1%})")
    print()

# Best / worst age groups
best_age  = age_stats['mean_per'].idxmin()
worst_age = age_stats['mean_per'].idxmax()
print(f"Best age group  : {best_age} (PER={age_stats.loc[best_age,'mean_per']:.4f}, n={age_stats.loc[best_age,'n']:.0f})")
print(f"Worst age group : {worst_age} (PER={age_stats.loc[worst_age,'mean_per']:.4f}, n={age_stats.loc[worst_age,'n']:.0f})")
print()

# Duration insight
if df['duration_bucket'].notna().any():
    dur_corr = df[['audio_duration_sec','per']].dropna().corr().iloc[0,1]
    print(f"Correlation(duration, PER) : {dur_corr:.4f}")

# Bias
print(f"Mean length diff (pred-gt) : {df['len_diff'].mean():+.3f} phones")
print(f"Fraction PER >= 1.0        : {(df['per'] >= 1.0).mean():.4f}")
print()

# Top 5 confusions
top5 = sub_df.head(5)
print("Top 5 phoneme substitutions:")
for _, r in top5.iterrows():
    print(f"  {r['gt_phone']} -> {r['pred_phone']}  ({r['count']} times)")

print("=" * 60)

## Length Mark Previous-Token Analysis

Find tokens immediately *before* length-marked tokens (ː or ˑ), then summarize frequency and probability.

In [ ]:
from collections import Counter
import json
import unicodedata
from pathlib import Path

length_marks = {'ː', 'ˑ'}

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'data').exists():
            return p
    return start

repo_root = find_repo_root(Path.cwd())

data_paths = [
    repo_root / 'data' / 'train_phon_transcripts.jsonl',
    repo_root / 'data' / 'train_phon_transcripts_talkbank.jsonl',
    repo_root / 'data' / 'train_phon_transcripts_ultrasuite.jsonl',
]

found_paths = [p for p in data_paths if p.exists()]
if not found_paths:
    raise FileNotFoundError(
        'No transcript files found. Expected one of: ' + ', '.join(str(p) for p in data_paths)
    )
if len(found_paths) < len(data_paths):
    missing = [p for p in data_paths if not p.exists()]
    print('Warning: missing files:', ', '.join(str(p) for p in missing))

def prev_symbol_before_length_mark(token: str, idx: int):
    j = idx - 1
    while j >= 0 and unicodedata.combining(token[j]):
        j -= 1
    if j >= 0:
        return token[j]
    return None

def analyze_prev_symbol(path: Path):
    prev_counts = Counter()
    mark_type_counts = Counter()
    total_marks = 0
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            tokens = obj.get('phonetic_text', '').split()
            for tok in tokens:
                for i, ch in enumerate(tok):
                    if ch in length_marks:
                        mark_type_counts[ch] += 1
                        total_marks += 1
                        prev = prev_symbol_before_length_mark(tok, i)
                        if prev is not None:
                            prev_counts[prev] += 1
    return prev_counts, total_marks, mark_type_counts

results = {}
for path in found_paths:
    results[path.name] = analyze_prev_symbol(path)

combined_prev = Counter()
combined_total = 0
combined_mark_types = Counter()
for prev_counts, total_marks, mark_types in results.values():
    combined_prev.update(prev_counts)
    combined_total += total_marks
    combined_mark_types.update(mark_types)

print('Combined length-mark occurrences:', combined_total)
print('Combined mark types:', dict(combined_mark_types))

print('Top 20 previous symbols:')
for sym, count in combined_prev.most_common(20):
    prob = count / combined_total if combined_total else 0.0
    print(sym, count, f'{prob:.4f}')

# Write outputs
out_dir = repo_root / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / 'length_mark_prev_symbol.csv'
with out_path.open('w', encoding='utf-8') as f:
    f.write('symbol,count,probability')
    for sym, count in combined_prev.most_common():
        prob = count / combined_total if combined_total else 0.0
        f.write(f'{sym},{count},{prob:.8f}')

per_path = out_dir / 'length_mark_prev_symbol_by_dataset.csv'
with per_path.open('w', encoding='utf-8') as f:
    f.write('dataset,symbol,count,probability')
    for name, (prev_counts, total_marks, _) in results.items():
        for sym, count in prev_counts.most_common():
            prob = count / total_marks if total_marks else 0.0
            f.write(f'{name},{sym},{count},{prob:.8f}')

seq_path = out_dir / 'length_mark_prev_symbol_sequences.csv'
with seq_path.open('w', encoding='utf-8') as f:
    f.write('dataset,utterance_id,token,mark_index,prev_symbol')
    for path in found_paths:
        with path.open('r', encoding='utf-8') as f_in:
            for line in f_in:
                obj = json.loads(line)
                tokens = obj.get('phonetic_text', '').split()
                for tok in tokens:
                    for i, ch in enumerate(tok):
                        if ch in length_marks:
                            prev = prev_symbol_before_length_mark(tok, i)
                            if prev is not None:
                                f.write(f'{path.name},{obj.get("utterance_id","")},{tok},{i},{prev}')

## Length Mark vs Predictions

Compare previous single-symbol distributions for length marks between ground truth and predictions, and quantify missing/extra length marks.

In [ ]:
import unicodedata
from collections import Counter
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Constants
LENGTH_MARKS = {'ː', 'ˑ'}


def get_symbol_before_mark(token: str, mark_idx: int) -> Optional[str]:
    """Finds the first non-combining character preceding a length mark."""
    j = mark_idx - 1
    while j >= 0 and unicodedata.combining(token[j]):
        j -= 1
    return token[j] if j >= 0 else None


def count_preceding_symbols(text_series: pd.Series) -> Counter:
    """Counts symbols appearing immediately before length marks in a series."""
    counts = Counter()
    for text in text_series.dropna():
        for token in str(text).split():
            for i, char in enumerate(token):
                if char in LENGTH_MARKS:
                    prev_sym = get_symbol_before_mark(token, i)
                    if prev_sym is not None:
                        counts[prev_sym] += 1
    return counts


def get_alignment_operations(source: List[str], target: List[str]) -> List[Tuple[str, str, str]]:
    """
    Computes Levenshtein distance and returns a list of edit operations:
    (operation, source_char, target_char)
    """
    m, n = len(source), len(target)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if source[i-1] == target[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j-1],  # Substitute
                    dp[i-1][j],    # Delete
                    dp[i][j-1]     # Insert
                )

    # Backtrack to find operations
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and source[i-1] == target[j-1]:
            i -= 1; j -= 1  # Match
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(('sub', source[i-1], target[j-1]))
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(('del', source[i-1], '_'))
            i -= 1
        else:
            ops.append(('ins', '_', target[j-1]))
            j -= 1
            
    return ops[::-1]


def calculate_alignment_metrics(df: pd.DataFrame) -> Dict[str, int]:
    """Calculates missing, extra, and substituted length marks via alignment."""
    metrics = {
        'gt_total': 0, 'pred_total': 0,
        'missing': 0, 'extra': 0, 'substituted': 0
    }

    for gt, pred in zip(df['ground_truth'], df['prediction']):
        gt_chars = list(str(gt).replace(' ', ''))
        pred_chars = list(str(pred).replace(' ', ''))

        metrics['gt_total'] += sum(1 for c in gt_chars if c in LENGTH_MARKS)
        metrics['pred_total'] += sum(1 for c in pred_chars if c in LENGTH_MARKS)

        for op, gt_c, pred_c in get_alignment_operations(gt_chars, pred_chars):
            if op == 'del' and gt_c in LENGTH_MARKS:
                metrics['missing'] += 1
            elif op == 'ins' and pred_c in LENGTH_MARKS:
                metrics['extra'] += 1
            elif op == 'sub':
                if gt_c in LENGTH_MARKS and pred_c in LENGTH_MARKS and gt_c != pred_c:
                    metrics['substituted'] += 1
                elif gt_c in LENGTH_MARKS and pred_c not in LENGTH_MARKS:
                    metrics['missing'] += 1
                elif pred_c in LENGTH_MARKS and gt_c not in LENGTH_MARKS:
                    metrics['extra'] += 1

    metrics['matched'] = metrics['gt_total'] - metrics['missing'] - metrics['substituted']
    return metrics


def plot_analysis_results(compare_df: pd.DataFrame, metrics: Dict[str, int], out_path: Path):
    """Generates and saves visualizations for the length mark analysis."""
    
    # --- Plot 1: Top 15 Symbols Distribution ---
    top_df = compare_df.head(15)
    
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    x = np.arange(len(top_df['symbol']))
    width = 0.35
    
    ax1.bar(x - width/2, top_df['gt_count'], width, label='Ground Truth', color='#2ca02c')
    ax1.bar(x + width/2, top_df['pred_count'], width, label='Prediction', color='#1f77b4')
    
    ax1.set_ylabel('Counts')
    ax1.set_title('Top Symbols Preceding Length Marks (GT vs Pred)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(top_df['symbol'])
    ax1.legend()
    
    fig1.tight_layout()
    fig1.savefig(out_path / 'length_mark_symbols_plot.png')
    plt.close(fig1)

    # --- Plot 2: Error Distribution Breakdown ---
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    categories = ['matched', 'missing', 'extra', 'substituted']
    counts = [metrics[cat] for cat in categories]
    
    # Different colors for matched vs. errors
    colors = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd']
    
    bars = ax2.bar(categories, counts, color=colors)
    ax2.set_ylabel('Count')
    ax2.set_title('Length Mark Alignment Summary')
    
    # Add data labels on top of the bars
    for bar in bars:
        height = bar.get_height()
        ax2.annotate(f'{height}',
                     xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3),  # 3 points vertical offset
                     textcoords="offset points",
                     ha='center', va='bottom')

    fig2.tight_layout()
    fig2.savefig(out_path / 'length_mark_errors_plot.png')
    plt.close(fig2)


def analyze_length_marks(df: pd.DataFrame, output_dir: str = 'outputs'):
    """Main orchestrator function to analyze length marks, plot, and save output."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    # --- 1. Symbol Distributions ---
    gt_counts = count_preceding_symbols(df['ground_truth'])
    pred_counts = count_preceding_symbols(df['prediction'])

    symbols = sorted(set(gt_counts) | set(pred_counts))
    gt_total = sum(gt_counts.values())
    pred_total = sum(pred_counts.values())

    compare_rows = []
    for sym in symbols:
        gt = gt_counts.get(sym, 0)
        pr = pred_counts.get(sym, 0)
        compare_rows.append({
            'symbol': sym,
            'gt_count': gt,
            'gt_prob': gt / gt_total if gt_total else 0.0,
            'pred_count': pr,
            'pred_prob': pr / pred_total if pred_total else 0.0,
            'diff_count': pr - gt,
        })

    compare_df = pd.DataFrame(compare_rows).sort_values(['gt_count', 'pred_count'], ascending=False)
    
    print("--- Symbol Distributions ---")
    display(compare_df.head(20))
    compare_df.to_csv(out_path / 'length_mark_prev_symbol_gt_vs_pred.csv', index=False)

    # --- 2. Error Alignment ---
    metrics = calculate_alignment_metrics(df)

    print("\n--- Length Mark Alignment Summary ---")
    print(f"GT length marks:       {metrics['gt_total']}")
    print(f"Pred length marks:     {metrics['pred_total']}")
    print(f"Matched marks:         {metrics['matched']}")
    print(f"Missing (GT not pred): {metrics['missing']}")
    print(f"Extra (pred not in GT):{metrics['extra']}")
    print(f"Substitutions (ː ↔ ˑ): {metrics['substituted']}")

    metrics_df = pd.DataFrame([{'metric': k, 'value': v} for k, v in metrics.items()])
    metrics_df.to_csv(out_path / 'length_mark_error_summary.csv', index=False)
    
    # --- 3. Generate Plots ---
    plot_analysis_results(compare_df, metrics, out_path)
    print(f"\nPlots and CSVs saved to: {out_path.absolute()}")

# ==========================================
# Execution
# ==========================================
# Assuming 'df' is already loaded in your environment:
# analyze_length_marks(df)


# --- Add this updated function inside your script to ensure plots show up ---
def plot_analysis_results(compare_df: pd.DataFrame, metrics: Dict[str, int], out_path: Path):
    """Generates, saves, and DISPLAYS visualizations."""
    
    # 1. Top Symbols Plot
    top_df = compare_df.head(15)
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    x = np.arange(len(top_df['symbol']))
    width = 0.35
    ax1.bar(x - width/2, top_df['gt_count'], width, label='Ground Truth', color='#2ca02c')
    ax1.bar(x + width/2, top_df['pred_count'], width, label='Prediction', color='#1f77b4')
    ax1.set_ylabel('Counts')
    ax1.set_title('Top Symbols Preceding Length Marks (GT vs Pred)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(top_df['symbol'])
    ax1.legend()
    fig1.tight_layout()
    fig1.savefig(out_path / 'length_mark_symbols_plot.png')
    plt.show() # <--- THIS ENSURES IT APPEARS IN THE NOTEBOOK

    # 2. Error Distribution Plot
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    categories = ['matched', 'missing', 'extra', 'substituted']
    counts = [metrics[cat] for cat in categories]
    colors = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd']
    bars = ax2.bar(categories, counts, color=colors)
    ax2.set_ylabel('Count')
    ax2.set_title('Length Mark Alignment Summary')
    for bar in bars:
        height = bar.get_height()
        ax2.annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    fig2.tight_layout()
    fig2.savefig(out_path / 'length_mark_errors_plot.png')
    plt.show() # <--- THIS ENSURES IT APPEARS IN THE NOTEBOOK

# ==========================================
# FINAL STEP: CALL THE FUNCTION
# ==========================================
# Make sure your dataframe is named 'df' and has the right columns!
if 'df' in locals():
    analyze_length_marks(df)
else:
    print("Error: The variable 'df' was not found. Please load your data first.")

## PER If Length Marks Were Perfect

Estimate PER if all length-mark (ː/ˑ) errors were corrected while keeping other errors unchanged.

In [ ]:
length_marks = {'ː', 'ˑ'}

def alignment_ops(a: list, b: list):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(('sub', a[i-1], b[j-1]))
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(('del', a[i-1], '_'))
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            ops.append(('ins', '_', b[j-1]))
            j -= 1
    return ops

orig_pers = []
new_pers = []

for g, p in zip(df['ground_truth'], df['prediction']):
    g_chars = list(str(g).replace(' ', ''))
    p_chars = list(str(p).replace(' ', ''))
    gt_len = len(g_chars)
    if gt_len == 0:
        continue
    ops = alignment_ops(g_chars, p_chars)
    edit_dist = len(ops)
    lenmark_ops = 0
    for op, gc, pc in ops:
        if op == 'del' and gc in length_marks:
            lenmark_ops += 1
        elif op == 'ins' and pc in length_marks:
            lenmark_ops += 1
        elif op == 'sub' and (gc in length_marks or pc in length_marks):
            lenmark_ops += 1
    new_edit = max(edit_dist - lenmark_ops, 0)
    orig_pers.append(edit_dist / gt_len)
    new_pers.append(new_edit / gt_len)

orig_mean = sum(orig_pers) / len(orig_pers)
new_mean = sum(new_pers) / len(new_pers)

print(f"Original mean PER (macro): {orig_mean:.6f}")
print(f"Mean PER if length marks perfect: {new_mean:.6f}")
print(f"Absolute PER reduction: {orig_mean - new_mean:.6f}")

In [ ]:
# --- Compare Parquet Output Files ---
# Check if two parquet files have the exact same utterance IDs.

import pandas as pd
from pathlib import Path

# Resolve repo root to ensure relative paths work from the notebook's directory
repo_root = Path.cwd().resolve()
for parent in [repo_root] + list(repo_root.parents):
    if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
        repo_root = parent
        break

# TODO: Update these paths to the actual parquet files you want to compare
path_a = repo_root / "outputs/2026-03-16/17-21-53_screaming-conquest-127/oof_predictions_best.parquet" 
path_b = repo_root / "outputs/2026-03-14/10-17-34_worst-turfje-109/oof_predictions_best.parquet"

if path_a.exists() and path_b.exists():
    df_a = pd.read_parquet(path_a)
    df_b = pd.read_parquet(path_b)
    
    print(f"File A shape: {df_a.shape}")
    print(f"File B shape: {df_b.shape}")
    
    if 'utterance_id' in df_a.columns and 'utterance_id' in df_b.columns:
        ids_a = set(df_a['utterance_id'])
        ids_b = set(df_b['utterance_id'])
        
        print(f"Common utterance IDs: {len(ids_a & ids_b)}")
        print(f"IDs only in File A: {len(ids_a - ids_b)}")
        print(f"IDs only in File B: {len(ids_b - ids_a)}")
        
        if len(ids_a) == len(ids_b) and ids_a == ids_b:
            print("✅ Both files contain the exact same utterance IDs!")
        else:
            print("❌ The utterance IDs differ between the two files.")
    else:
        print("Column 'utterance_id' not found in one or both dataframes. Available columns:")
        print(f"A: {df_a.columns.tolist()}")
        print(f"B: {df_b.columns.tolist()}")
else:
    print(f"One or both files not found. Please verify the following paths:")
    print(f"path_a: {path_a.resolve()} - Exists: {path_a.exists()}")
    print(f"path_b: {path_b.resolve()} - Exists: {path_b.exists()}")

## Phoneme Frequency Comparison (GT vs Predictions)

In [ ]:
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Count all phonemes (characters) in ground truth and predictions
gt_all_phones = Counter()
pred_all_phones = Counter()

for gt, pred in zip(df['ground_truth'], df['prediction']):
    # Do NOT remove spaces anymore, so we capture the " " token
    gt_text = str(gt)
    pred_text = str(pred)
    
    for char in gt_text:
        gt_all_phones[char] += 1
    
    for char in pred_text:
        pred_all_phones[char] += 1

# Get all unique phonemes
all_phonemes = sorted(set(gt_all_phones.keys()) | set(pred_all_phones.keys()))

# Create comparison dataframe
phoneme_comparison = pd.DataFrame({
    'phoneme': all_phonemes,
    'gt_count': [gt_all_phones[p] for p in all_phonemes],
    'pred_count': [pred_all_phones[p] for p in all_phonemes],
})

# Calculate raw diff and percentage diff
phoneme_comparison['diff'] = phoneme_comparison['pred_count'] - phoneme_comparison['gt_count']
phoneme_comparison['diff_pct'] = np.where(
    phoneme_comparison['gt_count'] > 0,
    (phoneme_comparison['diff'] / phoneme_comparison['gt_count']) * 100,
    np.nan  # If GT is 0, percentage is undefined
)

# ---------------------------------------------------------------------------------
# TODO: Replace `vocab_dict` with your actual tokenizer vocabulary map. 
# For example: vocab_dict = processor.tokenizer.get_vocab()
# ---------------------------------------------------------------------------------
vocab_dict = {' ': 0} # Placeholder: putting space first as an example

# Map each phoneme to its tokenizer ID. (If missing, assign a high number like 9999)
phoneme_comparison['token_id'] = phoneme_comparison['phoneme'].map(lambda x: vocab_dict.get(x, 9999))

# Sort by the tokenizer ID
phoneme_comparison = phoneme_comparison.sort_values('gt_count', ascending=False).reset_index(drop=True)

print(f"Total unique phonemes: {len(all_phonemes)}")
print("\nTop phonemes by ground truth frequency (Sorted by Tokenizer ID):")
display(phoneme_comparison.head(55).drop(columns=['token_id']).style.format({
    'gt_count': '{:,}', 
    'pred_count': '{:,}',
    'diff': '{:+,}',
    'diff_pct': '{:+.2f}%'
}).background_gradient(subset=['diff_pct'], cmap='RdYlGn_r', vmin=-50, vmax=50))

# Create visualization
fig, ax = plt.subplots(figsize=(16, 8))

# Take top 55 phonemes by frequency for the chart
top_n = 55
top_phonemes = phoneme_comparison.head(top_n)

x = np.arange(len(top_phonemes))
width = 0.35

bars1 = ax.bar(x - width/2, top_phonemes['gt_count'], width, label='Ground Truth', color='#2ca02c', alpha=0.8)
bars2 = ax.bar(x + width/2, top_phonemes['pred_count'], width, label='Predictions', color='#1f77b4', alpha=0.8)

ax.set_xlabel('Phoneme', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'Phoneme Frequency Comparison: Ground Truth vs Predictions (Top {top_n})', fontsize=14, fontweight='bold')

# Replace literal ' ' with '[SPACE]' for readability on the x-axis
x_labels = [p if p != ' ' else '[SPACE]' for p in top_phonemes['phoneme']]
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)

ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels on top of bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nTotal GT phonemes: {phoneme_comparison['gt_count'].sum():,}")
print(f"Total Pred phonemes: {phoneme_comparison['pred_count'].sum():,}")